In [1]:
import xarray as xr
from datetime import datetime
import math
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray as rio

In [2]:
CAM5_ex = xr.open_dataset('/nesi/project/niwa00015/queenle/data/CAM5-1-1/tas_A3hr_CAM5-1-1degree_All-Hist_est1_v2-0_run001_195901010000-195912312100.nc')

In [6]:
CAM5_ex.time

<xarray.DataArray 'time' (time: 2920)>
array([cftime.DatetimeNoLeap(1959, 1, 1, 0, 0, 0, 0, 6, 1),
       cftime.DatetimeNoLeap(1959, 1, 1, 3, 0, 0, 0, 6, 1),
       cftime.DatetimeNoLeap(1959, 1, 1, 6, 0, 0, 0, 6, 1), ...,
       cftime.DatetimeNoLeap(1959, 12, 31, 15, 0, 0, 0, 6, 365),
       cftime.DatetimeNoLeap(1959, 12, 31, 18, 0, 0, 0, 6, 365),
       cftime.DatetimeNoLeap(1959, 12, 31, 21, 0, 0, 0, 6, 365)], dtype=object)
Coordinates:
  * time     (time) object 1959-01-01 00:00:00 ... 1959-12-31 21:00:00
Attributes:
    axis:           T
    bounds:         time_bnds
    long_name:      time
    standard_name:  time

In [7]:
'''
LOAD DATA
'''

# ERA5 land-sea mask
lsm = xr.open_dataset('/nesi/project/niwa00015/queenle/data/ERA5/static/land_sea_mask.nc')
lsm = lsm.isel(time=0).drop('time')
lsm.rio.write_crs("epsg:4326", inplace=True)

# time zone shape file
tz_data = gpd.read_file("/nesi/project/niwa00015/queenle/data/time_zones/ne_10m_time_zones.shp")

# hourly ERA5 data
ERA5_hourly_ds = xr.open_dataset("/nesi/project/niwa00015/queenle/data/ERA5/hourly/post_79/ERA5-hourly-1979_2021.nc")


In [9]:
ERA5_tasmax = ERA5_hourly_ds.t2m.resample(time="1d").max().where(tz1_mask.lsm>0, other=0)
ERA5_tasmax = ERA5_tasmax.rename('tasmax')


NameError: name 'tz1_mask' is not defined

In [8]:
ERA5_hourly_ds

<xarray.Dataset>
Dimensions:    (latitude: 89, longitude: 77, time: 376944)
Coordinates:
  * time       (time) datetime64[ns] 1979-01-01 ... 2021-12-31T23:00:00
  * longitude  (longitude) float32 -129.0 -128.8 -128.5 ... -110.5 -110.2 -110.0
  * latitude   (latitude) float32 52.0 51.75 51.5 51.25 ... 30.5 30.25 30.0
Data variables:
    i10fg      (time, latitude, longitude) float64 ...
    tp         (time, latitude, longitude) float64 ...
    u10        (time, latitude, longitude) float64 ...
    v10        (time, latitude, longitude) float64 ...
    d2m        (time, latitude, longitude) float64 ...
    t2m        (time, latitude, longitude) float64 ...
Attributes:
    CDI:          Climate Data Interface version 2.0.5 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Thu Nov 24 17:08:59 2022: cdo -b F64 mergetime ERA5-hourly-...
    CDO:          Climate Data Operators version 2.0.5 (https://mpimet.mpg.de...

In [3]:
'''
CREATE LAND/TIME ZONE MASKS
'''

tz1 = tz_data.loc[tz_data.tz_name1st=='America/Los_Angeles'].geometry #zone=-8.0; tz_name1st = America/Los_Angeles
tz1_offset = tz_data.loc[tz_data.tz_name1st=='America/Los_Angeles'].zone.values[0]
tz1_noon_utc = 12 - tz1_offset # 12UTC - (-8) = 20UTC = local noon tz1 (PST)
tz1_mask = lsm.rio.clip(tz1, tz1.crs,drop=False).fillna(0)

tz2 = tz_data.loc[tz_data.tz_name1st=='America/Denver'].geometry #zone=-7.0; tz_name1st = America/Denver
tz2_offset = tz_data.loc[tz_data.tz_name1st=='America/Denver'].zone.values[0]
tz2_noon_utc = 12 - tz2_offset # 12UTC - (-7) = 19UTC = local noon tz2 (MST)
tz2_mask = lsm.rio.clip(tz2, tz2.crs,drop=False).fillna(0)

In [4]:
'''
For each time zone: select or sum over (tp) 24h intervals from local noon, find daily temperature max
'''

'''
Time zone 1: PST
'''
# accumulated hourly total precipitation 24 hours prior to local noon
tz1_tp = ERA5_hourly_ds.tp.resample(time="24H", base = tz1_noon_utc, closed="right",label="right").sum().where(tz1_mask.lsm>0, other=0)
tz1_tp = tz1_tp.resample(time='1D').first()

# select local noon for non precipitation values
tz1_non_tp_vars = ERA5_hourly_ds.drop('tp')
tz1_non_tp_noon = tz1_non_tp_vars.resample(time="24H", base = tz1_noon_utc, closed="right",label="right").last().where(tz1_mask.lsm>0, other=0)
tz1_non_tp_noon = tz1_non_tp_noon.resample(time='1D').first()

# select maximum temperature value between local noon + 1 previous day to local noon current day
tz1_t2m_max = ERA5_hourly_ds.t2m.resample(time="24H", base = 0, closed="right",label="right").max().where(tz1_mask.lsm>0, other=0)
tz1_t2m_max = tz1_t2m_max.rename('t2m_max')
tz1_t2m_max = tz1_t2m_max.resample(time='1D').first()

tz1_FWI_inputs = xr.merge([tz1_non_tp_noon,tz1_tp,tz1_t2m_max])

'''
Time zone 2: MST
'''
# accumulated hourly total precipitation 24 hours prior to local noon
tz2_tp = ERA5_hourly_ds.tp.resample(time="24H", base = tz2_noon_utc, closed="right",label="right").sum().where(tz2_mask.lsm>0,other=0)
tz2_tp = tz2_tp.resample(time='1D').first()

# select local noon for non precipitation values
tz2_non_tp_vars = ERA5_hourly_ds.drop('tp')
tz2_non_tp_noon = tz2_non_tp_vars.resample(time="24H", base = tz2_noon_utc, closed="right",label="right").last().where(tz2_mask.lsm>0, other=0)
tz2_non_tp_noon = tz2_non_tp_noon.resample(time='1D').first()

# select maximum temperature value between local noon + 1 previous day to local noon current day
tz2_t2m_max = ERA5_hourly_ds.t2m.resample(time="24H", base = 0, closed="right",label="right").max().where(tz2_mask.lsm>0, other=0)
tz2_t2m_max = tz2_t2m_max.rename('t2m_max')
tz2_t2m_max = tz2_t2m_max.resample(time='1D').first()

tz2_FWI_inputs = xr.merge([tz2_non_tp_noon,tz2_tp,tz2_t2m_max])

'''
Combine two local-noon datasets together to single dataset
'''

FWI_inputs = tz1_FWI_inputs + tz2_FWI_inputs
FWI_inputs = FWI_inputs.where((tz1_mask + tz2_mask).lsm > 0)


In [9]:
'''
CONVERT UNITS
'''

FWI_inputs["t2m"] = FWI_inputs["t2m"] - 273.15 # convert temp K to C
FWI_inputs["d2m"] = FWI_inputs["d2m"] - 273.15 # convert dew point temp K to C
FWI_inputs["t2m_max"] = FWI_inputs["t2m_max"] - 273.15 # convert max temp K to C
FWI_inputs["tp"] = FWI_inputs["tp"] * 1000 # convert accumulated precip m to mm

FWI_inputs = FWI_inputs.assign(rh = 100*np.exp((243.04*17.625*(FWI_inputs["d2m"]-FWI_inputs["t2m"]))/((243.04+FWI_inputs["t2m"])*(243.04+FWI_inputs["d2m"]))))
FWI_inputs = FWI_inputs.assign(wind = 3.6 * np.sqrt(FWI_inputs["u10"]**2 + FWI_inputs["v10"]**2))


In [10]:
'''
Remove variables not relevant for FWI calculation
'''

FWI_inputs = FWI_inputs.drop(['u10','v10','d2m'])

In [11]:
'''
Write new netcdf file with FWI input variables to be used by calc_FWI program
'''

FWI_inputs.to_netcdf("/nesi/project/niwa00015/queenle/data/fire/FWI_inputs.nc")

In [30]:
'''
test that resampling is doing what I expect it to do
'''
print('TP')
print('resampled TP (41,-124) at ')
print(tz1_tp.sel(latitude=41,longitude=-124).isel(time=72).time.values)
print(tz1_tp.sel(latitude=41,longitude=-124).isel(time=72).values.tolist())
print()
print('manual selection and sum of (41,-124) 1979-03-13T21:00:00 to 1979-03-14T20:00:00')
print(ERA5_hourly_ds.tp.sel(latitude=41,longitude=-124,time=slice('1979-03-13T21:00:00','1979-03-14T20:00:00')).sum().values.tolist())
print('\n\n')

print('T2M')
print('resampled T2M (41,-124) at')
print(tz1_non_tp_noon.sel(latitude=41,longitude=-124).isel(time=1).time.values)
print(tz1_non_tp_noon.t2m.sel(latitude=41,longitude=-124).isel(time=1).values.tolist())
print()
print('manual selection of (41,-124) at 1979-01-02T20:00:00')
print(ERA5_hourly_ds.t2m.sel(latitude=41,longitude=-124,time='1979-01-02T20:00:00').values.tolist())
print('\n\n')

print('TMAX')
print('resampled TMAX (41,-124) at')
print(tz1_t2m_max.sel(latitude=41,longitude=-124).isel(time=1).time.values)
print(tz1_t2m_max.sel(latitude=41,longitude=-124).isel(time=1).values.tolist())
print()
print('manual selection of (41,-124) at 1979-01-01T21:00:00 to 1979-01-02T20:00:00')
print(ERA5_hourly_ds.t2m.sel(latitude=41,longitude=-124,time=slice('1979-01-01T21:00:00','1979-01-02T20:00:00')).max().values.tolist())

TP
resampled TP (41,-124) at 
1979-03-14T00:00:00.000000000
0.00026335348140013316

manual selection and sum of (41,-124) 1979-03-13T21:00:00 to 1979-03-14T20:00:00
0.0002633534814001331



T2M
resampled T2M (41,-124) at
1979-01-02T00:00:00.000000000
279.99257786449715

manual selection of (41,-124) at 1979-01-02T20:00:00
279.99257786449715



TMAX
resampled TMAX (41,-124) at
1979-01-02T00:00:00.000000000
283.4682055408962

manual selection of (41,-124) at 1979-01-01T21:00:00 to 1979-01-02T20:00:00
283.4682055408962
